# Imports and Paths


In [ ]:
# Imports

import json
import os
from pathlib import Path

import cv2
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd

from IPython.display import Image as DisplayImage, Video, display
from PIL import Image, ImageDraw
from tqdm.auto import tqdm


In [ ]:
# Paths

PROJECT_ROOT = Path.cwd()

LUNAR_EXPERT_VIDEO_DIR = PROJECT_ROOT / "lunarlander_expert" / "videos" / "expert_300"
GENERATED_VIDEO_DIR = PROJECT_ROOT / "generated_videos"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
PRETRAIN_DIR = ARTIFACTS_DIR / "action_pretrain"

CONTROL_POINTS_DIR = PRETRAIN_DIR / "control_points"
BACKGROUND_POINTS_DIR = PRETRAIN_DIR / "background_points"
ACTION_EMBEDDINGS_DIR = PRETRAIN_DIR / "action_embeddings"
MODELS_DIR = PRETRAIN_DIR / "models"

N_PRETRAIN_VIDEOS = 20
MAX_FRAMES_PER_EPISODE = 80

for path in [
    GENERATED_VIDEO_DIR,
    ARTIFACTS_DIR,
    PRETRAIN_DIR,
    CONTROL_POINTS_DIR,
    BACKGROUND_POINTS_DIR,
    ACTION_EMBEDDINGS_DIR,
    MODELS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

lunar_video_paths = sorted(LUNAR_EXPERT_VIDEO_DIR.glob("*.mp4"))[:N_PRETRAIN_VIDEOS]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LUNAR_EXPERT_VIDEO_DIR:", LUNAR_EXPERT_VIDEO_DIR)
print("num lunar videos:", len(lunar_video_paths))
print("PRETRAIN_DIR:", PRETRAIN_DIR)

In [ ]:
# Video helpers

def read_video(path, max_frames=None, stride=1, start_frame=0):
    cap = cv2.VideoCapture(str(path))

    if start_frame:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))

    frames_out = []
    frame_no = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_no % stride == 0:
            frames_out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if max_frames is not None and len(frames_out) >= max_frames:
                break

        frame_no += 1

    cap.release()

    if not frames_out:
        raise RuntimeError(f"No frames read from {path}")

    return np.stack(frames_out)


def sample_even_frame_indices(num_frames, max_frames):
    if num_frames <= 0:
        return np.asarray([], dtype=int)

    if max_frames is None or num_frames <= max_frames:
        return np.arange(num_frames, dtype=int)

    return np.unique(
        np.linspace(0, num_frames - 1, int(max_frames), dtype=int)
    )


def read_video_even(path, max_frames=None, stride=1, return_indices=False):
    full_frames = read_video(path, max_frames=None, stride=stride)
    frame_indices = sample_even_frame_indices(len(full_frames), max_frames)
    frames = full_frames[frame_indices]

    if return_indices:
        return frames, frame_indices

    return frames


def video_frame_count(path):
    return len(read_video(path, max_frames=None, stride=1))


def clip_indices(num_frames, clip_frames=48, seed=0):
    if num_frames <= 0:
        return np.asarray([], dtype=int)

    clip_frames = min(int(clip_frames), int(num_frames))
    rng = np.random.default_rng(seed)

    max_start = max(0, int(num_frames) - clip_frames)
    start = int(rng.integers(0, max_start + 1)) if max_start else 0

    return np.arange(start, start + clip_frames, dtype=int)


def save_video(frames, path, fps=30):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = frames.astype(np.uint8)
    imageio.mimsave(path, frames, fps=fps)
    return path


def show_video(frames, name, fps=30, embed=True):
    path = save_video(frames, GENERATED_VIDEO_DIR / name, fps=fps)
    display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path


def show_video_file(path, embed=True):
    path = Path(path)
    display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path

# Lunar Lander Videos


In [ ]:
# Lunar Lander expert videos

random_video_path = np.random.choice(lunar_video_paths)
random_video_frames = read_video(random_video_path)

print("num lunar expert videos:", len(lunar_video_paths))
print("random video:", random_video_path)
print("random video frames:", random_video_frames.shape)

show_video_file(random_video_path, embed=True)


# Control Points


## Model


In [ ]:
# Control Points - Model

CONTROL_POINT_NAMES = [
    "centroid",
    "axis_front",
    "axis_back",
    "axis_left",
    "axis_right",
    "contact_low",
]


def mask_to_record(mask, object_id):
    ys, xs = np.nonzero(mask)

    if len(xs) == 0:
        return None

    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1

    return {
        "object_id": object_id,
        "mask": mask,
        "box": (x1, y1, x2, y2),
        "area": int(mask.sum()),
        "centroid": (float(xs.mean()), float(ys.mean())),
    }


def largest_connected_component(mask, min_area=30):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8),
        connectivity=8,
    )

    if num_labels <= 1:
        return np.zeros_like(mask, dtype=bool)

    areas = stats[1:, cv2.CC_STAT_AREA]
    best = int(np.argmax(areas)) + 1

    if areas[best - 1] < min_area:
        return np.zeros_like(mask, dtype=bool)

    return labels == best


def lunar_lander_agent_records(frame):
    rgb = frame.astype(np.int16)
    brightness = rgb.mean(axis=2)
    colorfulness = np.max(rgb, axis=2) - np.min(rgb, axis=2)

    h, w = frame.shape[:2]
    yy, xx = np.indices((h, w))

    mask = (
        ((brightness > 70) | (colorfulness > 55))
        & (yy < int(h * 0.70))
        & (xx > int(w * 0.20))
        & (xx < int(w * 0.80))
    )

    mask = cv2.morphologyEx(
        mask.astype(np.uint8),
        cv2.MORPH_CLOSE,
        np.ones((3, 3), dtype=np.uint8),
        iterations=1,
    ).astype(bool)

    mask = largest_connected_component(mask, min_area=25)
    record = mask_to_record(mask, 0)

    if record is None:
        return []

    record.update(
        {
            "class_id": -101,
            "class_name": "lunar_lander_agent",
            "confidence": 1.0,
            "source": "env_specific_color_geometry",
        }
    )

    return [record]


def active_agent_records(frame, model=None, conf_min=0.25, max_objects=40, scene_type="lunar_lander"):
    if scene_type == "lunar_lander":
        return lunar_lander_agent_records(frame)

    raise ValueError(f"Unsupported scene_type: {scene_type}")


def render_active_agents_inverse_background(frame, records, title=None, alpha_bg=0.35, alpha_agent=0.55):
    active_mask = np.zeros(frame.shape[:2], dtype=bool)

    for record in records:
        active_mask |= record["mask"]

    bg_mask = ~active_mask
    image = frame.copy()

    image[bg_mask] = (
        image[bg_mask] * (1 - alpha_bg)
        + np.array([40, 180, 90]) * alpha_bg
    ).astype(np.uint8)

    image[active_mask] = (
        image[active_mask] * (1 - alpha_agent)
        + np.array([230, 50, 40]) * alpha_agent
    ).astype(np.uint8)

    return image, bg_mask, active_mask


def control_points_from_record(record):
    mask = record["mask"]
    ys, xs = np.nonzero(mask)

    if len(xs) == 0:
        return []

    x1, y1, x2, y2 = record["box"]
    centroid_x, centroid_y = record["centroid"]

    front_idx = np.argmin(xs)
    back_idx = np.argmax(xs)
    left_idx = np.argmax(ys)
    right_idx = np.argmin(ys)
    contact_idx = np.argmax(ys)

    points = [
        ("centroid", centroid_x, centroid_y),
        ("axis_front", float(xs[front_idx]), float(ys[front_idx])),
        ("axis_back", float(xs[back_idx]), float(ys[back_idx])),
        ("axis_left", float(xs[left_idx]), float(ys[left_idx])),
        ("axis_right", float(xs[right_idx]), float(ys[right_idx])),
        ("contact_low", float(xs[contact_idx]), float(ys[contact_idx])),
    ]

    rows = []

    for point_name, x, y in points:
        rows.append(
            {
                "point_name": point_name,
                "x": x,
                "y": y,
                "box_x1": x1,
                "box_y1": y1,
                "box_x2": x2,
                "box_y2": y2,
                "area": record["area"],
                "class_name": record["class_name"],
                "source": record["source"],
            }
        )

    return rows



## Test


In [ ]:
# Control Points - Test Video

test_video_path = np.random.choice(lunar_video_paths)
test_frames, test_frame_indices = read_video_even(
    test_video_path,
    max_frames=MAX_FRAMES_PER_EPISODE,
    return_indices=True,
)

annotated_frames = []
cp_test_rows = []

for local_frame_idx, frame in enumerate(test_frames):
    source_frame_idx = int(test_frame_indices[local_frame_idx])
    records = lunar_lander_agent_records(frame)

    cp_rows = []
    for track_id, obj in enumerate(records):
        rows = control_points_from_record(obj)

        for row in rows:
            row.update(
                {
                    "episode_id": 0,
                    "video_path": str(test_video_path),
                    "frame": source_frame_idx,
                    "local_frame": local_frame_idx,
                    "track_id": track_id,
                }
            )
            cp_test_rows.append(row)

        cp_rows.extend(rows)

    image = frame.copy()

    for row in cp_rows:
        x = int(round(row["x"]))
        y = int(round(row["y"]))

        cv2.circle(image, (x, y), 4, (255, 0, 0), -1)
        cv2.putText(
            image,
            row["point_name"],
            (x + 3, y + 3),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.35,
            (255, 255, 0),
            1,
            cv2.LINE_AA,
        )

    annotated_frames.append(image)

cp_test_df = pd.DataFrame(cp_test_rows)

print("test video:", test_video_path)
print("original frames:", video_frame_count(test_video_path))
print("sampled frames:", len(test_frames))
print("first sampled frame:", int(test_frame_indices[0]))
print("last sampled frame:", int(test_frame_indices[-1]))
print("sampled frame indices:", test_frame_indices.tolist())
print("cp_test_df:", cp_test_df.shape)

show_video(
    np.asarray(annotated_frames),
    "control_points_test_lunar_lander.mp4",
    fps=10,
    embed=True,
)

cp_test_df

## Loop


In [ ]:
# Control Points - Loop over selected Lunar Lander expert videos

control_point_rows = []

for episode_id, video_path in enumerate(tqdm(lunar_video_paths, desc="control points")):
    frames, frame_indices = read_video_even(
        video_path,
        max_frames=MAX_FRAMES_PER_EPISODE,
        return_indices=True,
    )

    for local_frame_idx, frame in enumerate(frames):
        source_frame_idx = int(frame_indices[local_frame_idx])
        records = lunar_lander_agent_records(frame)

        for track_id, record in enumerate(records):
            cp_rows = control_points_from_record(record)

            for row in cp_rows:
                row.update(
                    {
                        "episode_id": episode_id,
                        "video_path": str(video_path),
                        "frame": source_frame_idx,
                        "local_frame": local_frame_idx,
                        "track_id": track_id,
                    }
                )
                control_point_rows.append(row)

lunar_cp_df = pd.DataFrame(control_point_rows)

lunar_cp_df["point_name"] = pd.Categorical(
    lunar_cp_df["point_name"],
    categories=CONTROL_POINT_NAMES,
    ordered=True,
)

lunar_cp_df = lunar_cp_df.sort_values(
    ["episode_id", "track_id", "frame", "point_name"]
).reset_index(drop=True)

print("lunar_cp_df:", lunar_cp_df.shape)
print(lunar_cp_df.head())

## Save


In [ ]:
# Control Points - Save

control_points_output_path = CONTROL_POINTS_DIR / "lunar_lander_control_points.csv"

lunar_cp_df.to_csv(control_points_output_path, index=False)

print("saved:", control_points_output_path)
print("shape:", lunar_cp_df.shape)


# Background Segmentation


## Model


In [ ]:
# Background Points - Model

BACKGROUND_GRID_STEP = 32
BACKGROUND_DILATE_PX = 7
BACKGROUND_TEST_MAX_FRAMES = MAX_FRAMES_PER_EPISODE
BACKGROUND_TEST_FPS = 10


def background_mask_to_record(mask, object_id):
    ys, xs = np.nonzero(mask)

    if len(xs) == 0:
        return None

    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1

    return {
        "object_id": object_id,
        "mask": mask,
        "box": (x1, y1, x2, y2),
        "area": int(mask.sum()),
        "centroid": (float(xs.mean()), float(ys.mean())),
    }


def background_largest_connected_component(mask, min_area=30):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8),
        connectivity=8,
    )

    if num_labels <= 1:
        return np.zeros_like(mask, dtype=bool)

    areas = stats[1:, cv2.CC_STAT_AREA]
    best_label = int(np.argmax(areas)) + 1

    if areas[best_label - 1] < min_area:
        return np.zeros_like(mask, dtype=bool)

    return labels == best_label


def background_lunar_lander_agent_records(frame):
    rgb = frame.astype(np.int16)
    brightness = rgb.mean(axis=2)
    colorfulness = np.max(rgb, axis=2) - np.min(rgb, axis=2)

    height, width = frame.shape[:2]
    yy, xx = np.indices((height, width))

    agent_mask = (
        ((brightness > 70) | (colorfulness > 55))
        & (yy < int(height * 0.70))
        & (xx > int(width * 0.20))
        & (xx < int(width * 0.80))
    )

    agent_mask = cv2.morphologyEx(
        agent_mask.astype(np.uint8),
        cv2.MORPH_CLOSE,
        np.ones((3, 3), dtype=np.uint8),
        iterations=1,
    ).astype(bool)

    agent_mask = background_largest_connected_component(agent_mask, min_area=25)
    record = background_mask_to_record(agent_mask, object_id=0)

    if record is None:
        return []

    record.update(
        {
            "class_id": -101,
            "class_name": "lunar_lander_agent",
            "confidence": 1.0,
            "source": "env_specific_color_geometry",
        }
    )

    return [record]


def background_active_agent_records(frame, conf_min=0.25, scene_type="lunar_lander"):
    if scene_type != "lunar_lander":
        raise ValueError(f"Unsupported scene_type: {scene_type}")

    return background_lunar_lander_agent_records(frame)


def surface_grid_background_mask(
    frame,
    conf_min=0.25,
    dilate_px=BACKGROUND_DILATE_PX,
    scene_type="lunar_lander",
):
    records = background_active_agent_records(
        frame,
        conf_min=conf_min,
        scene_type=scene_type,
    )

    active_mask = np.zeros(frame.shape[:2], dtype=bool)

    for record in records:
        active_mask |= record["mask"]

    if dilate_px > 0 and active_mask.any():
        kernel = np.ones((dilate_px, dilate_px), dtype=np.uint8)
        active_mask = cv2.dilate(
            active_mask.astype(np.uint8),
            kernel,
            iterations=1,
        ).astype(bool)

    background_mask = ~active_mask

    return background_mask, active_mask, records


def make_surface_grid_points(mask, step=BACKGROUND_GRID_STEP, margin=12):
    height, width = mask.shape
    points = []

    for y in range(margin, height - margin, step):
        for x in range(margin, width - margin, step):
            if mask[y, x]:
                points.append((float(x), float(y)))

    return np.asarray(points, dtype=np.float32)


def add_missing_surface_grid_points(
    background_mask,
    points,
    valid,
    step=BACKGROUND_GRID_STEP,
    min_distance_ratio=0.72,
):
    candidate_points = make_surface_grid_points(background_mask, step=step)

    if len(candidate_points) == 0:
        return points.reshape(-1, 2), valid.astype(bool)

    points = points.reshape(-1, 2)
    valid = valid.astype(bool)
    existing_points = points[valid] if len(points) else np.empty((0, 2), dtype=np.float32)
    new_points = []
    min_dist_sq = float(step * min_distance_ratio) ** 2

    for candidate in candidate_points:
        if len(existing_points):
            nearest_existing = np.min(np.sum((existing_points - candidate) ** 2, axis=1))
            if nearest_existing < min_dist_sq:
                continue

        if new_points:
            added_points = np.asarray(new_points, dtype=np.float32)
            nearest_added = np.min(np.sum((added_points - candidate) ** 2, axis=1))
            if nearest_added < min_dist_sq:
                continue

        new_points.append(candidate)

    if not new_points:
        return points, valid

    new_points = np.asarray(new_points, dtype=np.float32)
    points = np.vstack([points, new_points]) if len(points) else new_points
    valid = np.concatenate([valid, np.ones(len(new_points), dtype=bool)])

    return points, valid


def track_surface_grid_points(
    frames_batch,
    indices,
    step=BACKGROUND_GRID_STEP,
    scene_type="lunar_lander",
    source_indices=None,
):
    if len(indices) == 0:
        return []

    if source_indices is None:
        source_indices = indices

    source_indices = np.asarray(source_indices, dtype=int)

    first_frame = frames_batch[int(indices[0])]
    background_mask, active_mask, records = surface_grid_background_mask(
        first_frame,
        scene_type=scene_type,
    )
    points = make_surface_grid_points(background_mask, step=step)
    valid = np.ones(len(points), dtype=bool)

    tracks = [
        {
            "frame": int(source_indices[0]),
            "points": points.copy(),
            "valid": valid.copy(),
            "prev_valid": np.zeros(len(points), dtype=bool),
            "dx": np.zeros(len(points), dtype=np.float32),
            "dy": np.zeros(len(points), dtype=np.float32),
            "speed": np.zeros(len(points), dtype=np.float32),
            "bg_mask": background_mask,
            "active_mask": active_mask,
            "records": records,
        }
    ]

    prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_RGB2GRAY)
    prev_points = points.reshape(-1, 1, 2)
    prev_valid = valid.copy()

    for local_pos, frame_idx in enumerate(indices[1:], start=1):
        source_frame_idx = int(source_indices[local_pos])
        frame = frames_batch[int(frame_idx)]
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        background_mask, active_mask, records = surface_grid_background_mask(
            frame,
            scene_type=scene_type,
        )

        if len(prev_points) == 0:
            next_points = prev_points.copy()
            valid = np.zeros(0, dtype=bool)
        else:
            next_points, status, _ = cv2.calcOpticalFlowPyrLK(
                prev_gray,
                gray,
                prev_points,
                None,
                winSize=(21, 21),
                maxLevel=3,
                criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
            )

            if next_points is None or status is None:
                next_points = prev_points.copy()
                valid = np.zeros(len(prev_points), dtype=bool)
            else:
                next_xy = next_points.reshape(-1, 2)
                height, width = background_mask.shape

                inside_frame = (
                    (next_xy[:, 0] >= 0)
                    & (next_xy[:, 0] < width)
                    & (next_xy[:, 1] >= 0)
                    & (next_xy[:, 1] < height)
                )

                on_background = np.zeros(len(next_xy), dtype=bool)
                rounded = np.floor(next_xy[inside_frame]).astype(int)
                rounded[:, 0] = np.clip(rounded[:, 0], 0, width - 1)
                rounded[:, 1] = np.clip(rounded[:, 1], 0, height - 1)
                on_background[inside_frame] = background_mask[rounded[:, 1], rounded[:, 0]]

                valid = (
                    prev_valid
                    & (status.reshape(-1) == 1)
                    & inside_frame
                    & on_background
                )

        tracked_xy = next_points.reshape(-1, 2)
        dx = tracked_xy[:, 0] - prev_points.reshape(-1, 2)[:, 0] if len(tracked_xy) else np.zeros(0, dtype=np.float32)
        dy = tracked_xy[:, 1] - prev_points.reshape(-1, 2)[:, 1] if len(tracked_xy) else np.zeros(0, dtype=np.float32)
        speed = np.sqrt(dx ** 2 + dy ** 2)

        tracked_xy, valid = add_missing_surface_grid_points(
            background_mask,
            tracked_xy,
            valid,
            step=step,
        )

        added_count = len(tracked_xy) - len(dx)
        if added_count > 0:
            dx = np.concatenate([dx, np.zeros(added_count, dtype=np.float32)])
            dy = np.concatenate([dy, np.zeros(added_count, dtype=np.float32)])
            speed = np.concatenate([speed, np.zeros(added_count, dtype=np.float32)])
            prev_valid = np.concatenate([prev_valid, np.zeros(added_count, dtype=bool)])

        tracks.append(
            {
                "frame": source_frame_idx,
                "points": tracked_xy.copy(),
                "valid": valid.copy(),
                "prev_valid": prev_valid.copy(),
                "dx": dx.astype(np.float32),
                "dy": dy.astype(np.float32),
                "speed": speed.astype(np.float32),
                "bg_mask": background_mask,
                "active_mask": active_mask,
                "records": records,
            }
        )

        prev_gray = gray
        prev_points = tracked_xy.reshape(-1, 1, 2)
        prev_valid = valid.copy()

    return tracks


def render_surface_grid_frame(frame, track, prev_track=None):
    image = frame.copy()
    background_mask = track["bg_mask"]
    active_mask = track["active_mask"]

    image[background_mask] = (
        image[background_mask] * 0.72
        + np.array([35, 175, 105]) * 0.28
    ).astype(np.uint8)
    image[active_mask] = (
        image[active_mask] * 0.35
        + np.array([230, 45, 35]) * 0.65
    ).astype(np.uint8)

    points = track["points"]
    valid = track["valid"]
    previous_points = None

    if prev_track is not None and len(prev_track["points"]) == len(points):
        previous_points = prev_track["points"]

    for point_id, (x, y) in enumerate(points):
        if not valid[point_id]:
            continue

        x_i, y_i = int(round(x)), int(round(y))

        if previous_points is not None:
            prev_x, prev_y = previous_points[point_id]
            cv2.arrowedLine(
                image,
                (int(round(prev_x)), int(round(prev_y))),
                (x_i, y_i),
                (0, 230, 255),
                1,
                tipLength=0.25,
            )

        cv2.circle(image, (x_i, y_i), 3, (255, 235, 0), -1)
        cv2.circle(image, (x_i, y_i), 3, (0, 0, 0), 1)

    return image


def surface_grid_summary_row(track):
    valid = track["valid"]
    speed = track["speed"]
    dx = track["dx"]
    dy = track["dy"]

    return {
        "frame": int(track["frame"]),
        "agents": int(len(track["records"])),
        "grid_points_total": int(len(track["points"])),
        "grid_points_tracked": int(valid.sum()),
        "background_pixels": int(track["bg_mask"].sum()),
        "median_dx": float(np.median(dx[valid])) if valid.any() else np.nan,
        "median_dy": float(np.median(dy[valid])) if valid.any() else np.nan,
        "median_speed": float(np.median(speed[valid])) if valid.any() else np.nan,
    }


def surface_grid_summary(tracks):
    return pd.DataFrame([surface_grid_summary_row(track) for track in tracks])


def tracks_to_background_dataframe(tracks, episode_id, video_path):
    rows = []

    for track in tracks:
        points = track["points"]
        valid = track["valid"]
        prev_valid = track["prev_valid"]
        dx = track["dx"]
        dy = track["dy"]
        speed = track["speed"]

        for background_point_id, (x, y) in enumerate(points):
            rows.append(
                {
                    "episode_id": int(episode_id),
                    "video_path": str(video_path),
                    "frame": int(track["frame"]),
                    "background_point_id": int(background_point_id),
                    "x": float(x),
                    "y": float(y),
                    "dx": float(dx[background_point_id]),
                    "dy": float(dy[background_point_id]),
                    "speed": float(speed[background_point_id]),
                    "valid": bool(valid[background_point_id]),
                    "prev_valid": bool(prev_valid[background_point_id]),
                }
            )

    return pd.DataFrame(rows)


def show_surface_grid_tracking_experiment(
    frames_batch,
    frame_idx,
    title,
    video_name,
    clip_frames=None,
    seed=0,
    grid_step=BACKGROUND_GRID_STEP,
    scene_type="lunar_lander",
    source_indices=None,
):
    indices = (
        np.arange(len(frames_batch), dtype=int)
        if clip_frames is None
        else clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    )

    if source_indices is None:
        source_indices = indices

    source_indices = np.asarray(source_indices, dtype=int)

    tracks = track_surface_grid_points(
        frames_batch,
        indices,
        step=grid_step,
        scene_type=scene_type,
        source_indices=source_indices,
    )

    selected_pos = int(np.argmin(np.abs(source_indices - frame_idx))) if len(indices) else 0

    rendered_frames = [
        render_surface_grid_frame(
            frames_batch[int(index)],
            track,
            tracks[pos - 1] if pos > 0 else None,
        )
        for pos, (index, track) in enumerate(zip(indices, tracks))
    ]

    plt.figure(figsize=(8, 5))
    plt.imshow(rendered_frames[selected_pos])
    plt.title(title)
    plt.axis("off")
    plt.show()

    print(
        f"video frames: "
        f"{int(source_indices[0]) if len(source_indices) else 0}:"
        f"{int(source_indices[-1]) + 1 if len(source_indices) else 0}"
    )

    video_path = show_video(
        np.asarray(rendered_frames),
        video_name,
        fps=BACKGROUND_TEST_FPS,
    )

    summary_df = surface_grid_summary(tracks)
    summary_df["video_path"] = str(video_path)

    display(summary_df.head(12))

    return tracks, np.asarray(rendered_frames), summary_df

## Test


In [ ]:
# Background Points - Test

test_video_path = np.random.choice(lunar_video_paths)
frames, frame_indices = read_video_even(
    test_video_path,
    max_frames=MAX_FRAMES_PER_EPISODE,
    return_indices=True,
)
frame_idx = int(frame_indices[min(10, len(frame_indices) - 1)])
local_indices = np.arange(len(frames), dtype=int)

print("test video:", test_video_path)
print("original frames:", video_frame_count(test_video_path))
print("sampled frames:", frames.shape)
print("first sampled frame:", int(frame_indices[0]))
print("last sampled frame:", int(frame_indices[-1]))
print("sampled frame indices:", frame_indices.tolist())
print("frame_idx:", frame_idx)

bg_surface_grid_1_lunar_tracks, bg_surface_grid_1_lunar_frames, bg_surface_grid_1_lunar_data = show_surface_grid_tracking_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Background Surface Grid Tracking",
    video_name="surface_grid_1_lunar_pretrain_test.mp4",
    scene_type="lunar_lander",
    source_indices=frame_indices,
)

Check df for background points shape for 1 episode.

In [ ]:
# Background Points - Test DF for the same random episode

bg_test_df = tracks_to_background_dataframe(
    bg_surface_grid_1_lunar_tracks,
    episode_id=0,
    video_path=test_video_path,
)

print("test video:", test_video_path)
print("bg_test_df shape:", bg_test_df.shape)

## Loop + Save

In [ ]:
# Background Points - Loop + Save Per Episode

background_output_paths = []

for episode_id, video_path in enumerate(tqdm(lunar_video_paths, desc="background points")):
    frames, frame_indices = read_video_even(
        video_path,
        max_frames=MAX_FRAMES_PER_EPISODE,
        return_indices=True,
    )

    local_indices = np.arange(len(frames), dtype=int)

    tracks = track_surface_grid_points(
        frames,
        local_indices,
        step=BACKGROUND_GRID_STEP,
        scene_type="lunar_lander",
        source_indices=frame_indices,
    )

    episode_bckg_df = tracks_to_background_dataframe(
        tracks,
        episode_id=episode_id,
        video_path=video_path,
    ).sort_values(
        ["episode_id", "frame", "background_point_id"]
    ).reset_index(drop=True)

    episode_output_path = (
        BACKGROUND_POINTS_DIR
        / f"lunar_lander_background_points_episode_{episode_id:04d}.csv"
    )

    episode_bckg_df.to_csv(episode_output_path, index=False)
    background_output_paths.append(episode_output_path)

    print(
        "saved:",
        episode_output_path.name,
        "shape:",
        episode_bckg_df.shape,
    )

print("saved episode files:", len(background_output_paths))

# Data Review

In [ ]:
# Data Review

background_paths = sorted(
    BACKGROUND_POINTS_DIR.glob("lunar_lander_background_points_episode_*.csv")
)

lunar_bckg_df = pd.concat(
    [pd.read_csv(path) for path in background_paths],
    ignore_index=True,
)

lunar_bckg_df = lunar_bckg_df.sort_values(
    ["episode_id", "frame", "background_point_id"]
).reset_index(drop=True)

control_points_path = CONTROL_POINTS_DIR / "lunar_lander_control_points.csv"
lunar_cp_df = pd.read_csv(control_points_path)

lunar_cp_df = lunar_cp_df.sort_values(
    ["episode_id", "track_id", "frame", "point_name"]
).reset_index(drop=True)

print("background files:", len(background_paths))
print("lunar_bckg_df shape:", lunar_bckg_df.shape)
print("lunar_cp_df shape:", lunar_cp_df.shape)

In [ ]:
lunar_bckg_df.head()

In [ ]:
lunar_cp_df.head()

# Action Encoder

## Define Variables

### Control Points

In [ ]:
# Action Encoder - Define Variables - Control Points

CONTROL_POINT_NAMES = [
    "centroid",
    "axis_front",
    "axis_back",
    "axis_left",
    "axis_right",
    "contact_low",
]

lunar_cp_df_reduced = lunar_cp_df[
    [
        "episode_id",
        "frame",
        "track_id",
        "point_name",
        "x",
        "y",
        "box_x1",
        "box_y1",
        "box_x2",
        "box_y2",
    ]
].copy()

lunar_cp_df_reduced["point_name"] = pd.Categorical(
    lunar_cp_df_reduced["point_name"],
    categories=CONTROL_POINT_NAMES,
    ordered=True,
)

lunar_cp_df_reduced = lunar_cp_df_reduced.sort_values(
    ["episode_id", "track_id", "frame", "point_name"]
).reset_index(drop=True)

print("lunar_cp_df_reduced shape:", lunar_cp_df_reduced.shape)
print(lunar_cp_df_reduced.head().to_string(index=False))

### Background

In [ ]:
# Action Encoder - Define Variables - Background

lunar_bckg_df_reduced = lunar_bckg_df[
    [
        "episode_id",
        "frame",
        "background_point_id",
        "x",
        "y",
        "dx",
        "dy",
        "speed",
        "valid",
        "prev_valid",
    ]
].copy()

lunar_bckg_df_reduced = lunar_bckg_df_reduced.sort_values(
    ["episode_id", "frame", "background_point_id"]
).reset_index(drop=True)

print("lunar_bckg_df_reduced shape:", lunar_bckg_df_reduced.shape)
print(lunar_bckg_df_reduced.head().to_string(index=False))

In [ ]:
# Action Encoder - Select Fixed Background Points

N_BACKGROUND_POINTS = 100

lunar_bckg_df_selected = (
    lunar_bckg_df_reduced
    .sort_values(["episode_id", "frame", "background_point_id"])
    .groupby(["episode_id", "frame"], group_keys=False)
    .apply(
        lambda frame_df: frame_df.sample(
            n=min(N_BACKGROUND_POINTS, len(frame_df)),
            random_state=42,
        )
    )
    .sort_values(["episode_id", "frame", "background_point_id"])
    .reset_index(drop=True)
)

print("lunar_bckg_df_selected shape:", lunar_bckg_df_selected.shape)
print(
    lunar_bckg_df_selected
    .groupby(["episode_id", "frame"])
    .size()
    .describe()
)

# Action Embedding

## Sample

### Plot 10 Consecutive Frames

In [ ]:
# Action Embeddings - Sample - Plot 10 Consecutive Frames

sample_episode_id = np.random.choice(lunar_cp_df_reduced["episode_id"].unique())

episode_frames = sorted(
    set(lunar_cp_df_reduced.loc[
        lunar_cp_df_reduced["episode_id"] == sample_episode_id,
        "frame",
    ])
    & set(lunar_bckg_df_selected.loc[
        lunar_bckg_df_selected["episode_id"] == sample_episode_id,
        "frame",
    ])
)

start_pos = np.random.randint(0, max(1, len(episode_frames) - 10 + 1))
sample_frames = episode_frames[start_pos:start_pos + 10]

fig, ax = plt.subplots(10, 1, figsize=(8, 40))

for row_idx, frame in enumerate(sample_frames):
    frame_tracks = lunar_cp_df_reduced[
        (lunar_cp_df_reduced["episode_id"] == sample_episode_id)
        & (lunar_cp_df_reduced["frame"] == frame)
    ]["track_id"]

    if frame_tracks.empty:
        ax[row_idx].set_title(f"episode={sample_episode_id}, frame={frame} | no control points")
        ax[row_idx].axis("off")
        continue

    track_id = frame_tracks.iloc[0]

    frame_cp = lunar_cp_df_reduced[
        (lunar_cp_df_reduced["episode_id"] == sample_episode_id)
        & (lunar_cp_df_reduced["frame"] == frame)
        & (lunar_cp_df_reduced["track_id"] == track_id)
    ].copy()

    frame_bckg = lunar_bckg_df_selected[
        (lunar_bckg_df_selected["episode_id"] == sample_episode_id)
        & (lunar_bckg_df_selected["frame"] == frame)
    ].copy()

    ax[row_idx].scatter(
        frame_bckg["x"],
        frame_bckg["y"],
        s=12,
        alpha=0.35,
        label="selected background points",
    )

    ax[row_idx].scatter(
        frame_cp["x"],
        frame_cp["y"],
        s=90,
        c="red",
        label="control points",
    )

    for _, cp_row in frame_cp.iterrows():
        ax[row_idx].text(
            cp_row["x"] + 3,
            cp_row["y"] + 3,
            cp_row["point_name"],
            fontsize=8,
            color="red",
        )

    ax[row_idx].invert_yaxis()
    ax[row_idx].set_xlabel("x image coordinate")
    ax[row_idx].set_ylabel("y image coordinate")
    ax[row_idx].set_title(f"episode={sample_episode_id}, frame={frame}, track_id={track_id}")

ax[0].legend(loc="upper right")

plt.suptitle(
    f"Control Points and Selected Background Points Before Outer Product | "
    f"episode={sample_episode_id}, frames {sample_frames[0]}-{sample_frames[-1]}",
    fontsize=16,
    y=1.002,
)

plt.tight_layout()
plt.show()

print("sample_episode_id:", sample_episode_id)
print("sample_frames:", sample_frames)
print(
    "selected background points per sampled frame:",
    lunar_bckg_df_selected[
        (lunar_bckg_df_selected["episode_id"] == sample_episode_id)
        & (lunar_bckg_df_selected["frame"].isin(sample_frames))
    ]
    .groupby("frame")
    .size()
    .to_dict()
)

### Single Action Embedding

In [ ]:
# Action Embeddings - Sample - Single Action Embedding

sample_episode_id = np.random.choice(lunar_cp_df_reduced["episode_id"].unique())

common_frames = sorted(
    set(lunar_cp_df_reduced.loc[
        lunar_cp_df_reduced["episode_id"] == sample_episode_id,
        "frame",
    ])
    & set(lunar_bckg_df_selected.loc[
        lunar_bckg_df_selected["episode_id"] == sample_episode_id,
        "frame",
    ])
)

frame = np.random.choice(common_frames)

frame_cp_all = lunar_cp_df_reduced[
    (lunar_cp_df_reduced["episode_id"] == sample_episode_id)
    & (lunar_cp_df_reduced["frame"] == frame)
]

track_id = frame_cp_all["track_id"].iloc[0]

frame_cp = frame_cp_all[
    frame_cp_all["track_id"] == track_id
].sort_values("point_name")

frame_bckg = lunar_bckg_df_selected[
    (lunar_bckg_df_selected["episode_id"] == sample_episode_id)
    & (lunar_bckg_df_selected["frame"] == frame)
].sort_values("background_point_id")

x_cp = frame_cp["x"]
y_cp = frame_cp["y"]

x_bckg = frame_bckg["x"]
y_bckg = frame_bckg["y"]

X_cp = x_cp.values.reshape(-1, 1)
Y_cp = y_cp.values.reshape(-1, 1)

X_bckg = x_bckg.values.reshape(-1, 1)
Y_bckg = y_bckg.values.reshape(-1, 1)

A_x = X_cp @ X_bckg.T
A_y = Y_cp @ Y_bckg.T

A = np.stack([A_x, A_y], axis=-1)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

im0 = ax[0].imshow(A_x, cmap="viridis", aspect="auto")
ax[0].set_title(f"$A^x$ | episode={sample_episode_id}, frame={frame}, track_id={track_id}")
ax[0].set_xlabel("selected background point index")
ax[0].set_ylabel("control point index")
fig.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(A_y, cmap="viridis", aspect="auto")
ax[1].set_title(f"$A^y$ | episode={sample_episode_id}, frame={frame}, track_id={track_id}")
ax[1].set_xlabel("selected background point index")
ax[1].set_ylabel("control point index")
fig.colorbar(im1, ax=ax[1])

plt.suptitle(
    f"Action Embedding Outer Product | episode={sample_episode_id}, frame={frame}, track_id={track_id} | A shape={A.shape}",
    fontsize=16,
    y=1.02,
)
plt.tight_layout()
plt.show()

print("A_x shape:", A_x.shape)
print("A_y shape:", A_y.shape)
print("A shape:", A.shape)

### Normalize Single Action Embedding

In [ ]:
# Action Embeddings - Sample - Normalize Single Action Embedding

A_x_norm = (A_x - A_x.mean()) / (A_x.std() + 1e-8)
A_y_norm = (A_y - A_y.mean()) / (A_y.std() + 1e-8)

A_norm = np.stack([A_x_norm, A_y_norm], axis=-1)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

im0 = ax[0].imshow(A_x_norm, cmap="viridis", aspect="auto")
ax[0].set_title(
    f"Normalized $A^x$ | episode={sample_episode_id}, frame={frame}, track_id={track_id}"
)
ax[0].set_xlabel("selected background point index")
ax[0].set_ylabel("control point index")
fig.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(A_y_norm, cmap="viridis", aspect="auto")
ax[1].set_title(
    f"Normalized $A^y$ | episode={sample_episode_id}, frame={frame}, track_id={track_id}"
)
ax[1].set_xlabel("selected background point index")
ax[1].set_ylabel("control point index")
fig.colorbar(im1, ax=ax[1])

plt.suptitle(
    f"Normalized Action Embedding Outer Product | "
    f"episode={sample_episode_id}, frame={frame}, track_id={track_id} | "
    f"A_norm shape={A_norm.shape}",
    fontsize=16,
    y=1.02,
)
plt.tight_layout()
plt.show()

print("A_x_norm mean:", round(A_x_norm.mean(), 6))
print("A_x_norm std:", round(A_x_norm.std(), 6))
print("A_y_norm mean:", round(A_y_norm.mean(), 6))
print("A_y_norm std:", round(A_y_norm.std(), 6))
print("A_norm shape:", A_norm.shape)

### Full Action Embeddings with Normalization for 1 Video

In [ ]:
# Full Action Embeddings with Normalization for All Videos

lunar_action_embeddings = {}

episode_ids = sorted(lunar_cp_df_reduced["episode_id"].unique())

for episode_id in tqdm(episode_ids, desc="action embeddings"):
    common_frames = sorted(
        set(lunar_cp_df_reduced.loc[
            lunar_cp_df_reduced["episode_id"] == episode_id,
            "frame",
        ].unique())
        & set(lunar_bckg_df_selected.loc[
            lunar_bckg_df_selected["episode_id"] == episode_id,
            "frame",
        ].unique())
    )

    episode_embeddings = {}

    for frame in common_frames:
        frame_bckg = lunar_bckg_df_selected[
            (lunar_bckg_df_selected["episode_id"] == episode_id)
            & (lunar_bckg_df_selected["frame"] == frame)
        ].sort_values("background_point_id")

        X_bckg = frame_bckg[["x"]].to_numpy()
        Y_bckg = frame_bckg[["y"]].to_numpy()

        frame_embeddings = {}

        for track_id, frame_cp in lunar_cp_df_reduced[
            (lunar_cp_df_reduced["episode_id"] == episode_id)
            & (lunar_cp_df_reduced["frame"] == frame)
        ].groupby("track_id", sort=False):

            frame_cp = frame_cp.sort_values("point_name")

            X_cp = frame_cp[["x"]].to_numpy()
            Y_cp = frame_cp[["y"]].to_numpy()

            A_x = X_cp @ X_bckg.T
            A_y = Y_cp @ Y_bckg.T
            A = np.stack([A_x, A_y], axis=-1)

            A_min = A.min()
            A_max = A.max()
            A_norm = (A - A_min) / (A_max - A_min + 1e-8)

            A_x_norm = A_norm[:, :, 0]
            A_y_norm = A_norm[:, :, 1]

            frame_embeddings[track_id] = {
                "A_x": A_x,
                "A_y": A_y,
                "A": A,
                "A_min": A_min,
                "A_max": A_max,
                "A_x_norm": A_x_norm,
                "A_y_norm": A_y_norm,
                "A_norm": A_norm,
                "token_flat": A.reshape(-1),
                "token_norm_flat": A_norm.reshape(-1),
                "control_point_dim": len(frame_cp),
                "background_dim": len(frame_bckg),
                "relation_channels": A.shape[-1],
                "flattened_dim": A.reshape(-1).shape[0],
            }

        episode_embeddings[frame] = frame_embeddings

    lunar_action_embeddings[episode_id] = episode_embeddings

print("episodes:", len(lunar_action_embeddings))
print("episode_ids:", sorted(lunar_action_embeddings.keys()))

for episode_id, episode_embeddings in lunar_action_embeddings.items():
    print(
        f"episode_id={episode_id:>3} "
        f"frames={len(episode_embeddings):>4}"
    )

#### Change of Action Embedding

In [ ]:
# Action Embeddings - Sample - Temporal Change Plot

sample_episode_id = int(np.random.choice(sorted(lunar_action_embeddings.keys())))

available_track_ids = sorted({
    track_id
    for frame in lunar_action_embeddings[sample_episode_id].values()
    for track_id in frame.keys()
})

track_id = int(np.random.choice(available_track_ids))

track_frames = [
    frame
    for frame in sorted(lunar_action_embeddings[sample_episode_id])
    if track_id in lunar_action_embeddings[sample_episode_id][frame]
]

track_tokens = np.stack([
    lunar_action_embeddings[sample_episode_id][frame][track_id]["token_norm_flat"]
    for frame in track_frames
])

delta_norm = np.linalg.norm(np.diff(track_tokens, axis=0), axis=1)

plt.plot(track_frames[1:], delta_norm)
plt.xlabel("frame")
plt.ylabel("||A_norm_t - A_norm_t-1||")
plt.title(
    f"Normalized action embedding temporal change | "
    f"episode={sample_episode_id}, track_id={track_id}"
)
plt.show()

print("sample_episode_id:", sample_episode_id)
print("track_id:", track_id)
print("available episodes:", sorted(lunar_action_embeddings.keys()))
print("track_frames:", track_frames)

# Transformer Model

In [ ]:
# Transformer Model - Hyperparameters

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)

sample_episode_id = next(iter(lunar_action_embeddings))
sample_frame = next(iter(lunar_action_embeddings[sample_episode_id]))
sample_track_id = next(iter(lunar_action_embeddings[sample_episode_id][sample_frame]))
sample_embedding = lunar_action_embeddings[sample_episode_id][sample_frame][sample_track_id]

control_point_dim = sample_embedding["control_point_dim"]
background_dim = sample_embedding["background_dim"]
relation_channels = sample_embedding["relation_channels"]
flattened_dim = sample_embedding["flattened_dim"]

seq_len = 4
batch_size = 8
embed_dim = 256
n_heads = 4
n_blocks = 4
dropout = 0.1

assert embed_dim % n_heads == 0

{
    "device": device,
    "control_point_dim": control_point_dim,
    "background_dim": background_dim,
    "relation_channels": relation_channels,
    "flattened_dim": flattened_dim,
    "seq_len": seq_len,
    "batch_size": batch_size,
    "embed_dim": embed_dim,
    "n_heads": n_heads,
    "n_blocks": n_blocks,
    "dropout": dropout,
}

### Action Embedding Token Sequence

In [ ]:
# Transformer Model - Visualize Track Tokens

sample_episode_id = int(np.random.choice(sorted(lunar_action_embeddings.keys())))

available_track_ids = sorted({
    track_id
    for frame_embeddings in lunar_action_embeddings[sample_episode_id].values()
    for track_id in frame_embeddings.keys()
})

track_id = int(np.random.choice(available_track_ids))

track_frames = [
    frame
    for frame in sorted(lunar_action_embeddings[sample_episode_id])
    if track_id in lunar_action_embeddings[sample_episode_id][frame]
]

track_tokens = np.stack([
    lunar_action_embeddings[sample_episode_id][frame][track_id]["token_norm_flat"]
    for frame in track_frames
])

plt.figure(figsize=(12, 5))
plt.imshow(track_tokens, aspect="auto", cmap="viridis")
plt.xlabel("action embedding dimension")
plt.ylabel("frame index")
plt.title(
    f"Action embedding tokens | episode={sample_episode_id}, track_id={track_id}"
)
plt.colorbar()
plt.show()

print("sample_episode_id:", sample_episode_id)
print("track_id:", track_id)
print("track_tokens shape:", track_tokens.shape)

### Sequence Dataset

In [ ]:
# Transformer Model - Sequence Dataset

all_sequences = []

for episode_id in sorted(lunar_action_embeddings.keys()):
    available_track_ids = sorted({
        track_id
        for frame_embeddings in lunar_action_embeddings[episode_id].values()
        for track_id in frame_embeddings.keys()
    })

    for track_id in available_track_ids:
        track_frames = [
            frame
            for frame in sorted(lunar_action_embeddings[episode_id])
            if track_id in lunar_action_embeddings[episode_id][frame]
        ]

        if len(track_frames) <= seq_len:
            continue

        track_tokens = np.stack([
            lunar_action_embeddings[episode_id][frame][track_id]["A_norm"].reshape(-1)
            for frame in track_frames
        ]).astype(np.float32)

        all_sequences.append(
            {
                "episode_id": episode_id,
                "track_id": track_id,
                "frames": track_frames,
                "tokens": track_tokens,
            }
        )

split_idx = int(len(all_sequences) * 0.8)

train_sequences = all_sequences[:split_idx]
test_sequences = all_sequences[split_idx:]

def get_data_batch(train=True, batch_size=batch_size):
    sequences = train_sequences if train else test_sequences

    X_batch = []
    y_batch = []

    for _ in range(batch_size):
        seq = sequences[np.random.randint(0, len(sequences))]
        tokens = seq["tokens"]

        max_start = len(tokens) - seq_len - 1

        if max_start < 0:
            continue

        start = np.random.randint(0, max_start + 1)

        X_batch.append(tokens[start:start + seq_len])
        y_batch.append(tokens[start + 1:start + seq_len + 1])

    if len(X_batch) == 0:
        raise ValueError("No valid sequences for batch.")

    return (
        torch.tensor(np.stack(X_batch), dtype=torch.float32),
        torch.tensor(np.stack(y_batch), dtype=torch.float32),
    )

print("all_sequences:", len(all_sequences))
print("train_sequences:", len(train_sequences))
print("test_sequences:", len(test_sequences))
print("token shape:", all_sequences[0]["tokens"].shape)

### Attention and Transformer Block

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self):
    super().__init__()

    # number of attention heads
    self.num_heads = n_heads
    self.head_dim = embed_dim // n_heads

    # Q, K, V
    self.QKV = nn.Linear(embed_dim, 3*embed_dim, bias = True)

    # linear mixing after attention
    self.W0 = nn.Linear(embed_dim, embed_dim, bias = True)

    # n dropout not defined here b/c it's in F.scaled_dot_product_attention

  def forward(self, x):

    # sizes for later use
    B, T, E = x.shape # [batch, seq_len, embed_dim]

    # push data through QKV in one matrix
    qkv = self.QKV(x)
    q,k,v = torch.split(qkv,E,dim=2)

    # reshape to [B,T,nHeads, head_dim]

    q = q.view(B,T,self.num_heads, self.head_dim).transpose(1,2)
    k = k.view(B,T,self.num_heads, self.head_dim).transpose(1,2)
    v = v.view(B,T,self.num_heads, self.head_dim).transpose(1,2)

    dropp = dropout if self.training==True else 0
    out = F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=dropp)

    # recombine heads: (B, n_heads, T, head_dim) -> [B, T, E]
    # out = out.transpose(1,2).view(B,T,E)
    out = out.transpose(1, 2).contiguous().view(B, T, E)

    # finally, linearly mix the attention heads
    out = self.W0(out)

    return out

class TransformerBlock(nn.Module):
  def __init__(self):
    super().__init__()

    # attention subblock
    self.layernorm_1 = nn.LayerNorm(embed_dim, eps = 1e-5)
    self.attn = MultiHeadAttention()

    # linear feedforward (MLP) subblock
    self.layernorm_2 = nn.LayerNorm(embed_dim, eps = 1e-5)
    # 4x expansion then back
    self.mlp_1 = nn.Linear(embed_dim, 4*embed_dim, bias = True)
    self.gelu  = nn.GELU()
    self.mlp_2 = nn.Linear(4*embed_dim, embed_dim, bias = True)

    # n transformer block dropout
    self.trn_dropout = nn.Dropout(dropout)

  def forward(self, x):

    # attention
    x_att = self.layernorm_1(x) # pre-attention norm
    x_att = x + self.trn_dropout(self.attn(x_att)) # n attention -> dropout -> add

    # MLP
    x_ff = self.layernorm_2(x_att) # pre-MLP normalizaiton
    x_ff = self.mlp_2(self.gelu(self.mlp_1(x_ff)))
    x_ff = x_att + self.trn_dropout(x_ff)

    return x_ff

## Transformer Model Wrapper

In [ ]:
class ActionEmbeddingModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(flattened_dim, embed_dim, bias=True)
        self.wpe = nn.Embedding(seq_len, embed_dim)
        self.emb_dropout = nn.Dropout(dropout)

        self.transformerBlocks = nn.Sequential(
            *[TransformerBlock() for _ in range(n_blocks)]
        )

        self.layernorm_final = nn.LayerNorm(embed_dim, eps=1e-5)
        self.final_head = nn.Linear(embed_dim, flattened_dim, bias=True)

        self.apply(self.weightInits)

    def weightInits(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0, std=.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        if isinstance(module, nn.Embedding):
            nn.init.xavier_normal_(module.weight)

    def forward(self, z):
        # z: [B, T, flattened_dim]
        token_emb = self.input_proj(z)
        posit_emb = self.wpe(torch.arange(z.shape[1], device=z.device))

        x = token_emb + posit_emb
        x = self.emb_dropout(x)
        x = self.transformerBlocks(x)
        x = self.layernorm_final(x)
        out = self.final_head(x)

        return out


### Model Initialization

In [ ]:
model = ActionEmbeddingModel().to(device)

### Loss and Optimizer

In [ ]:
loss_function = nn.MSELoss().to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01,
)

X, y = get_data_batch()

pred = model(X.to(device))
loss = loss_function(pred[:, -1, :], y[:, -1, :].to(device))

loss

### Training Loop

In [ ]:
# Transformer Model - Training Loop

num_samples = 5000

train_loss = []
test_loss = []
test_steps = []

for sampli in range(num_samples):
    X, y = get_data_batch(train=True)

    model.zero_grad(set_to_none=True)

    pred = model(X.to(device))
    loss = loss_function(pred, y.to(device))

    loss.backward()
    optimizer.step()

    train_loss.append(loss.item())

    with torch.no_grad():
        model.eval()

        X_test, y_test = get_data_batch(train=False)
        pred_test = model(X_test.to(device))
        thisloss = loss_function(pred_test, y_test.to(device))

        test_loss.append(thisloss.item())
        test_steps.append(sampli)

        model.train()

    print(
        f"Sample {sampli:4}, "
        f"train loss: {train_loss[-1]:8.5f}, "
        f"test loss: {test_loss[-1]:8.5f}"
    )

### Losses

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(range(len(train_loss)), train_loss, label="train loss")
plt.plot(range(len(test_loss)), test_loss, marker="o", label="test loss")

plt.xlabel("training step")
plt.ylabel("MSE loss")
plt.title("Action Embedding Transformer Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Action Embeddings Evaluation

In [ ]:
with torch.no_grad():
    model.eval()

    X, y = get_data_batch(False)
    pred = model(X.to(device)).cpu()

    model.train()

sample_idx = 0

true_embedding = y[sample_idx, -1].numpy()
pred_embedding = pred[sample_idx, -1].numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(true_embedding, label="true")
ax[0].plot(pred_embedding, label="pred", alpha=0.75)
ax[0].set_title("Flattened Action Embedding: True vs Pred")
ax[0].set_xlabel("flattened embedding index")
ax[0].set_ylabel("value")
ax[0].legend()

ax[1].scatter(true_embedding, pred_embedding, s=8, alpha=0.5)
ax[1].set_title("Predicted vs True Embedding Values")
ax[1].set_xlabel("true")
ax[1].set_ylabel("pred")

plt.tight_layout()
plt.show()